## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1) A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)

2) A function or coroutine to execute.

In [5]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response




AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds known for their ability to mimic human speech. I\'ve heard they have strong vocal cords or maybe some other mechanism that allows them to copy sounds. But why do they do that? Is it just for communication within their species, or is there more to it?\n\nFirst, I should consider their natural behavior. In the wild, parrots live in groups, right? They must communicate with each other for various reasons like finding food, warning of predators, or social bonding. So maybe their ability to mimic sounds is an extension of their natural communication skills. They might use different calls to interact with others, and when they\'re in captivity, they might adapt to mimic human voices as a way to interact with humans.\n\nI remember reading that parrots have a part of their brain that\'s involved in vocal learning. Maybe other birds like songbirds also 

In [6]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

### call the tool

In [7]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted.\n', 'tool_calls': [{'id': '67xxyj14r', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 154, 'total_tokens': 239, 'completion_time': 0.123292204, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.006119562, 'prompt_tokens_details': None, 'queue_time': 0.052804067, 'total_time': 0.129411766}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e9450-9ce0-74e2-885f-c45

### Tool execution Loop

In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. Let me know if you'd like more details! ☀️


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': '368ec7c6k', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.147892046, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.007302523, 'prompt_tokens_details': None, 'queue_time': 0.053467167, 'total_time': 0.155194569}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_